# TP Jour 2 — Modèle d'Exécution Spark, Jointures Multi-Datasets & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Observer la **Lazy Evaluation** et distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Profiler la qualité de vos tables **Bronze** (issues d'un ou **plusieurs datasets Kaggle**), nettoyer les anomalies, typer fortement les colonnes et harmoniser les **clés de jointure inter-sources**.
3. Mettre en œuvre une **jointure distribuée multi-datasets maîtrisée** (`BroadcastHashJoin` via `F.broadcast` pour un référentiel/dataset compact, ou `SortMergeJoin` entre deux grandes sources) **en contrôlant la cardinalité (absence d'explosion N-à-M)**.
4. Sauvegarder votre table analytique enrichie dans **`/lakehouse/silver/`** et expérimenter les garanties **ACID de Delta Lake** (`_delta_log`, **Schema Enforcement**, **`MERGE INTO`**, **Time Travel**).

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J2_Spark_Delta_Silver")
    .getOrCreate()
)

print("Session Spark active — Spark UI : http://localhost:4040")

Session Spark active — Spark UI : http://localhost:4040


## 1. Chargement de vos tables Bronze (Multi-Datasets / Multi-Tables) & Diagnostic Qualité

### Exercice 1.1 — Charger toutes vos tables Bronze et profiler valeurs manquantes / doublons
1. Chargez dans `bronze_dfs` votre table de faits principale ainsi que la (ou les) table(s) secondaire(s) à joindre depuis `/lakehouse/bronze/<nom_table>`.
2. Comptez le nombre de valeurs `NULL` sur vos colonnes clés (en particulier les **clés de jointure**) ainsi que les doublons éventuels.

In [11]:
# Les chemins correspondent aux noms produits par la boucle Bronze du Jour 1.
BRONZE_TABLES: dict[str, str] = {
    "main": "/lakehouse/bronze/olist_orders_dataset",
    "items": "/lakehouse/bronze/olist_order_items_dataset",
    "products": "/lakehouse/bronze/olist_products_dataset",
    "reviews": "/lakehouse/bronze/olist_order_reviews_dataset",
}

bronze_dfs = {alias: spark.read.format("delta").load(path) for alias, path in BRONZE_TABLES.items()}
df_bronze = bronze_dfs["main"]

key_columns = {
    "main": [
        "order_id", "customer_id", "order_status", "order_purchase_timestamp",
        "order_delivered_customer_date", "order_estimated_delivery_date",
    ],
    "items": ["order_id", "order_item_id", "product_id", "price", "freight_value"],
    "products": ["product_id", "product_category_name", "product_weight_g"],
    "reviews": ["order_id", "review_id", "review_score", "review_comment_message"],
}
null_rows = []
for alias, df in bronze_dfs.items():
    columns = key_columns[alias]
    profile = df.agg(
        F.count("*").alias("total"),
        *[F.count(F.when(F.col(column).isNull(), 1)).alias(column) for column in columns],
    ).first()
    print(f"[{alias}] {BRONZE_TABLES[alias]} → {profile.total:,} lignes")
    null_rows.extend((alias, column, profile[column]) for column in columns)

null_report = spark.createDataFrame(null_rows, "table string, colonne string, nb_null long")
print("Valeurs NULL sur les colonnes clés")
null_report.orderBy("table", "colonne").show(len(null_rows), truncate=False)

# Les clés composites identifient un article ou un avis à l'intérieur d'une commande.
# Un order_id répété dans items ou reviews signale une relation un-à-plusieurs.
duplicate_keys = {
    "main": [("order_id",)],
    "items": [("order_id", "order_item_id"), ("order_id",)],
    "products": [("product_id",)],
    "reviews": [("order_id", "review_id"), ("order_id",)],
}
duplicate_rows = []
for alias, keys in duplicate_keys.items():
    for key in keys:
        groups = bronze_dfs[alias].dropna(subset=list(key)).groupBy(*key).count()
        summary = groups.where(F.col("count") > 1).agg(
            F.count("*").alias("groupes"),
            F.coalesce(F.sum(F.col("count") - 1), F.lit(0)).alias("lignes"),
        ).first()
        duplicate_rows.append((alias, " + ".join(key), summary.groupes, summary.lignes))

duplicate_report = spark.createDataFrame(
    duplicate_rows, "table string, cle string, groupes_repetes long, lignes_supplementaires long"
)
print("Clés répétées (clés NULL exclues)")
duplicate_report.orderBy("table", "cle").show(len(duplicate_rows), truncate=False)

[main] /lakehouse/bronze/olist_orders_dataset → 99,441 lignes
[items] /lakehouse/bronze/olist_order_items_dataset → 112,650 lignes
[products] /lakehouse/bronze/olist_products_dataset → 32,951 lignes
[reviews] /lakehouse/bronze/olist_order_reviews_dataset → 99,224 lignes
Valeurs NULL sur les colonnes clés
+--------+-----------------------------+-------+
|table   |colonne                      |nb_null|
+--------+-----------------------------+-------+
|items   |freight_value                |0      |
|items   |order_id                     |0      |
|items   |order_item_id                |0      |
|items   |price                        |0      |
|items   |product_id                   |0      |
|main    |customer_id                  |0      |
|main    |order_delivered_customer_date|2965   |
|main    |order_estimated_delivery_date|0      |
|main    |order_id                     |0      |
|main    |order_purchase_timestamp     |0      |
|main    |order_status                 |0      |
|product

## 2. Harmonisation Inter-Datasets, Nettoyage & Jointure Distribuée (`BroadcastHashJoin` / `SortMergeJoin`)

### Bonnes pratiques de jointure entre plusieurs datasets hétérogènes :
1. **Harmonisation des clés de jointure (*Key Alignment*)** : deux datasets Kaggle d'auteurs différents n'ont presque jamais exactement le même format de clé :
   - *Axe temporel* : ramenez les horodatages à la même granularité et au même fuseau (`F.to_date(...)`, `F.date_trunc("hour", ...)` ou format `yyyy-MM`).
   - *Axe géographique / catégoriel* : normalisez la casse, les espaces et les codes (`F.upper(F.trim(col))`, code ISO pays/région, code postal ou grille `F.round(lat, 2)`).
2. **Prévention des explosions N-à-M** : avant un `LEFT JOIN` de la table principale (`N` lignes) vers un second dataset, assurez-vous que la clé de jointure est **unique côté table secondaire** via une pré-agrégation métier (`groupBy(join_keys).agg(...)`) ou `dropDuplicates(join_keys)` ! Vérifiez systématiquement que le nombre de lignes après `LEFT JOIN` n'explose pas.
3. **Choix de la stratégie physique Spark** :
   - **Table secondaire compacte (< quelques dizaines de Mo)** : forcez **`F.broadcast(df_sec)`** pour obtenir un `BroadcastHashJoin` sans *Shuffle* de la grande table.
   - **Deux grands datasets volumineux** : laissez Spark exécuter un `SortMergeJoin` distribué (optimisé dynamiquement par *Adaptive Query Execution*).
  *(Si votre dataset Kaggle ne comporte qu'un seul fichier plat, construisez ou extrayez une table de référence/catégories dédiée pour pratiquer cette jointure et vérifier le plan physique !)*

### Exercice 2.1 — Nettoyer, harmoniser vos clés, joindre vos sources et écrire `/lakehouse/silver/<table_silver>`
1. Nettoyez vos tables (`dropDuplicates`, filtrage d'anomalies, typage fort `F.to_timestamp` / `F.to_date` / numériques) et harmonisez la clé de jointure commune.
2. Créez au moins **2 nouvelles colonnes métier dérivées** (dont idéalement un indicateur combinant vos deux sources !) utiles pour les Jours 3 & 4.
3. Réalisez la jointure (avec `F.broadcast(...)` sur une table de dimension/dataset secondaire compact), contrôlez le **taux de rapprochement (*match rate*)** ainsi que l'absence de duplication N-M, et affichez `.explain("simple")`.
4. Écrivez la table enrichie au format **Delta** dans **`/lakehouse/silver/<votre_table>`**.

**Application Olist.** On conserve les commandes livrées à un seul article. La cible `a_un_commentaire` vaut 1 si un commentaire écrit a été envoyé à partir de la livraison. La jointure gauche conserve les commandes sans avis.

In [12]:
SILVER_MAIN_PATH = "/lakehouse/silver/olist_commentaires_livraison"

orders = (
    df_bronze.where(F.col("order_status") == "delivered")
    .select(
        F.trim("order_id").alias("order_id"),
        F.to_timestamp("order_purchase_timestamp").alias("date_commande"),
        F.to_timestamp("order_delivered_customer_date").alias("date_livraison"),
        F.to_date("order_estimated_delivery_date").alias("date_livraison_prevue"),
    )
    .dropna()
    .where(F.col("date_livraison") >= F.col("date_commande"))
    .dropDuplicates()
)
items = bronze_dfs["items"].select(
    F.trim("order_id").alias("order_id"),
    F.col("order_item_id").cast("int").alias("order_item_id"),
    F.trim("product_id").alias("product_id"),
    F.col("price").cast("double").alias("prix"),
    F.col("freight_value").cast("double").alias("frais_livraison"),
).dropDuplicates()

# Compter les articles avant de filtrer les valeurs négatives.
single_item_orders = items.groupBy("order_id").count().where(F.col("count") == 1).select("order_id")
single_items = items.join(single_item_orders, "order_id", "left_semi").where(
    (F.col("prix") >= 0) & (F.col("frais_livraison") >= 0)
)
candidates = orders.join(single_items, "order_id", "inner").drop("order_item_id")
count_before_join = candidates.count()
assert count_before_join > 0, "Aucune commande livrée à un seul article après filtrage."

products = bronze_dfs["products"].select(
    F.trim("product_id").alias("product_id"),
    F.trim("product_category_name").alias("categorie_produit"),
    F.col("product_weight_g").cast("double").alias("poids_produit_g"),
    F.lit(True).alias("_produit_trouve"),
).dropDuplicates()
reviews = bronze_dfs["reviews"].select(
    F.trim("order_id").alias("order_id"),
    F.to_timestamp("review_answer_timestamp").alias("date_reponse"),
    F.col("review_comment_message").rlike(r"(?U)\S").alias("message_ecrit"),
)
# Au plus une ligne d'avis par commande, même si plusieurs réponses ont été envoyées.
labels = (
    reviews.join(candidates.select("order_id", "date_livraison"), "order_id")
    .groupBy("order_id")
    .agg(
        F.max(
            F.when(F.col("message_ecrit") & (F.col("date_reponse") >= F.col("date_livraison")), 1)
            .otherwise(0)
        ).alias("a_un_commentaire"),
        F.lit(True).alias("_avis_trouve"),
    )
)
enriched = candidates.join(F.broadcast(products), "product_id", "left").join(labels, "order_id", "left")
df_silver = (
    enriched.withColumn(
        "categorie_produit",
        F.when(F.length("categorie_produit") > 0, F.col("categorie_produit")).otherwise("inconnue"),
    )
    .withColumn("delai_livraison_jours", F.datediff("date_livraison", "date_commande"))
    .withColumn("retard_jours", F.greatest(F.datediff("date_livraison", "date_livraison_prevue"), F.lit(0)))
    .withColumn("livraison_en_retard", F.col("retard_jours") > 0)
    .withColumn("a_un_commentaire", F.coalesce("a_un_commentaire", F.lit(0)))
    .withColumn("_mis_a_jour_le", F.lit(None).cast("timestamp"))
    .drop("_produit_trouve", "_avis_trouve")
)

metrics = enriched.agg(
    F.count("*").alias("total"),
    F.countDistinct("order_id").alias("commandes_uniques"),
    F.count("_produit_trouve").alias("produits_trouves"),
    F.count("_avis_trouve").alias("avis_trouves"),
    F.count(F.when(F.col("a_un_commentaire") == 1, 1)).alias("commentaires"),
).first()
assert metrics.total == metrics.commandes_uniques == count_before_join, (
    "La jointure doit conserver exactement une ligne par commande."
)
print(f"Commandes avant/après jointures : {count_before_join:,} / {metrics.total:,}")
print(f"Rapprochement produits : {metrics.produits_trouves / metrics.total:.1%}")
print(f"Commandes avec au moins un avis : {metrics.avis_trouves / metrics.total:.1%}")
print(f"Commentaires écrits à partir de la livraison : {metrics.commentaires:,} ({metrics.commentaires / metrics.total:.1%})")
df_silver.explain("simple")
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(SILVER_MAIN_PATH)
silver_count = spark.read.format("delta").load(SILVER_MAIN_PATH).count()
assert silver_count == count_before_join
print(f"Table Silver Delta : {SILVER_MAIN_PATH} ({silver_count:,} lignes)")

Commandes avant/après jointures : 86,835 / 86,835
Rapprochement produits : 100.0%
Commandes avec au moins un avis : 99.4%
Commentaires écrits à partir de la livraison : 31,286 (36.0%)
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [order_id#8282, product_id#8288, date_commande#8283, date_livraison#8284, date_livraison_prevue#8285, prix#8289, frais_livraison#8290, categorie_produit#8600, poids_produit_g#8539, coalesce(a_un_commentaire#8544, 0) AS a_un_commentaire#8604, delai_livraison_jours#8601, retard_jours#8602, (retard_jours#8602 > 0) AS livraison_en_retard#8603, null AS _mis_a_jour_le#8605]
   +- Project [order_id#8282, product_id#8288, date_commande#8283, date_livraison#8284, date_livraison_prevue#8285, prix#8289, frais_livraison#8290, CASE WHEN (length(categorie_produit#8538) > 0) THEN categorie_produit#8538 ELSE inconnue END AS categorie_produit#8600, poids_produit_g#8539, a_un_commentaire#8544, datediff(cast(date_livraison#8284 as date), cast(date_commande#8

Table Silver Delta : /lakehouse/silver/olist_commentaires_livraison (86,835 lignes)


## 3. Garanties Transactionnelles ACID de Delta Lake

### Exercice 3.1 — Explorer `_delta_log` et vérifier le *Schema Enforcement*
1. Lisez et affichez les premières lignes du fichier JSON de transaction `00000000000000000000.json` situé dans `{SILVER_MAIN_PATH}/_delta_log/`.
2. Tentez d'insérer (`mode("append")`) un petit DataFrame artificiel au schéma incompatible dans `SILVER_MAIN_PATH` et interceptez l'exception levée par Delta Lake.

In [13]:
from pyspark.errors import AnalysisException

print("Premières lignes du journal Delta (version 0)")
spark.read.text(f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json").show(8, truncate=110)
target_delta = DeltaTable.forPath(spark, SILVER_MAIN_PATH)
version_before = target_delta.history(1).select("version").first().version
bad_df = spark.createDataFrame([("test_1", "colonne_intruse")], ["fake_id", "intruder_col"])
try:
    bad_df.write.format("delta").mode("append").option("mergeSchema", "false").save(SILVER_MAIN_PATH)
except AnalysisException as exc:
    message = str(exc).lower()
    if "schema" not in message or "mismatch" not in message:
        raise
    print("Schema Enforcement validé : Delta a rejeté le schéma incompatible.")
else:
    raise AssertionError("Delta a accepté le schéma incompatible.")

version_after = target_delta.history(1).select("version").first().version
assert version_after == version_before, "L'écriture rejetée ne doit créer aucun commit."
print(f"Version intacte après le rejet : {version_after}")

Premières lignes du journal Delta (version 0)
+--------------------------------------------------------------------------------------------------------------+
|                                                                                                         value|
+--------------------------------------------------------------------------------------------------------------+
|{"commitInfo":{"timestamp":1794914996514,"operation":"WRITE","operationParameters":{"mode":"Overwrite","par...|
|{"metaData":{"id":"4462a2b9-326e-4c85-bfa5-45bcf6f576ac","format":{"provider":"parquet","options":{}},"sche...|
|                                                      {"protocol":{"minReaderVersion":1,"minWriterVersion":2}}|
|{"add":{"path":"part-00000-71e66358-5158-4a96-8889-3cfd548dc350-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00001-d14fc365-aa59-4449-bc9d-85e1d57486a6-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00002-d65882b6-ed2e-49bc-b96

### Exercice 3.2 — Upsert (`MERGE INTO`) et Voyage dans le Temps (`VERSION AS OF`)
1. Prélevez un petit lot de lignes existantes dans votre table Silver (ou simulez un lot correctif + 1 nouvelle ligne) et appliquez un **`MERGE INTO`** via `DeltaTable.forPath(spark, SILVER_MAIN_PATH)` (`whenMatchedUpdate` / `whenNotMatchedInsert`).
2. Affichez l'historique des commits avec `DESCRIBE HISTORY` et interrogez la version initiale de la table via `.option("versionAsOf", 0)` pour démontrer l'auditabilité temporelle !

**Démonstration Olist.** Le `MERGE` renseigne la colonne d’audit `_mis_a_jour_le` sur une commande réelle. Le voyage temporel permet d’afficher sa valeur en version 0 et après le `MERGE`.

In [15]:
from datetime import datetime, timezone

target_delta = DeltaTable.forPath(spark, SILVER_MAIN_PATH)
correction = (
    target_delta.toDF().orderBy("order_id").limit(1)
    .withColumn("_mis_a_jour_le", F.lit(datetime.now(timezone.utc)).cast("timestamp"))
)
order_id_demo = correction.select("order_id").first().order_id

# Le MERGE ajoute une trace d'audit à une commande réelle.
(
    target_delta.alias("target")
    .merge(correction.alias("source"), "target.order_id = source.order_id")
    .whenMatchedUpdate(set={"_mis_a_jour_le": "source._mis_a_jour_le"})
    .whenNotMatchedInsertAll()
    .execute()
)

delta_history = target_delta.history().orderBy(F.desc("version"))
delta_history.select(
    "version", "timestamp", "operation",
    F.col("operationMetrics").getItem("numTargetRowsUpdated").alias("lignes_mises_a_jour"),
    F.col("operationMetrics").getItem("numTargetRowsInserted").alias("lignes_inserees"),
).show(10, truncate=False)

initial = spark.read.format("delta").option("versionAsOf", 0).load(SILVER_MAIN_PATH)
current = spark.read.format("delta").load(SILVER_MAIN_PATH)
avant = initial.where(F.col("order_id") == order_id_demo).select("_mis_a_jour_le").first()
apres = current.where(F.col("order_id") == order_id_demo).select("_mis_a_jour_le").first()
assert avant["_mis_a_jour_le"] is None and apres["_mis_a_jour_le"] is not None
print(f"Commande testée : {order_id_demo}")
print(f"Version 0 : {avant['_mis_a_jour_le']} | Version actuelle : {apres['_mis_a_jour_le']}")

26/11/17 14:27:07 WARN MapPartitionsRDD: RDD 1009 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


+-------+-----------------------+---------+-------------------+---------------+
|version|timestamp              |operation|lignes_mises_a_jour|lignes_inserees|
+-------+-----------------------+---------+-------------------+---------------+
|4      |2026-11-17 14:27:05.967|MERGE    |1                  |0              |
|3      |2026-11-17 11:46:58.799|MERGE    |1                  |0              |
|2      |2026-11-17 11:46:42.531|WRITE    |NULL               |NULL           |
|1      |2026-11-17 11:30:03.887|MERGE    |1                  |0              |
|0      |2026-11-17 11:29:56.64 |WRITE    |NULL               |NULL           |
+-------+-----------------------+---------+-------------------+---------------+



[Stage 510:=========================================>            (38 + 12) / 50]

Commande testée : 00010242fe8c5a6d1ba2dd792cb16214
Version 0 : None | Version actuelle : 2026-11-17 14:26:58.768950
